# 🫀 실험 21 — **두 번째 외부 병원**. 중국 코호트에서 rule-out 이 살아남나

**MedKOS / `notebooks/exp21_chapman_ningbo.ipynb`** · 퀘스트 `ailab-2026-0015`

PhysioNet **Open Access** · `ecg-arrhythmia` 1.0.0 (Chapman-Shaoxing + Ningbo, 45,152 레코드)
SNOMED-CT 라벨이라 **부위 세분화가 없다** → **MI 유무(MI-any)** 한 축만 본다.

## 왜 이 실험인가

실험20(PTBDB)은 독일 단일 기관이고 MI 유병률 49% 인 진단 코호트였다.
여기는 **중국 두 병원 · 다른 기기 · 다른 인종 · 부정맥 중심 코호트**다.
바뀌는 축이 완전히 다르므로 **실험20 의 발견이 우연이 아닌지 복제 검정**할 수 있다.

실험20b 가 세 가지를 남겼고, 이번에 전부 두 번째 데이터에서 다시 묻는다:

| 실험20/20b 가 찾은 것 | 실험21 에서 다시 묻는 것 |
|---|---|
| 외부에서 **12유도가 더 많이** 무너진다(D 음수) | 다른 병원에서도 복제되나 → **P-3** |
| 이 모델은 rule-in 이 아니라 **rule-out** 도구다(LR− 0.071) | rule-out 이 살아남나 → **P-2** |
| **동작점이 시드 간에 이전되지 않는다**(임계값 8.4배 차) | 앙상블 임계값이 이걸 고치나 → **P-4** |

## 사전등록

| 관문 | 내용 | 지지 조건 |
|---|---|---|
| **G0-a** | SNOMED 코드 전수 열거 + MI 코드 선정 | 미매핑 0 · **MI 레코드 ≥ 200** 아니면 중단 |
| **G0-b** | 유도 순서(아인트호벤) · 대역 정합 사다리 | 실험20 과 동일 규격 |
| **P-1** | MI-any 외부 AUROC 낙폭 | `< 0.10` |
| **P-2★★** | **rule-out 이 살아남나** — 동결 동작점의 외부 `LR−` | `LR− ≤ 0.20` |
| **P-3★★** | **복제** — 외부 잔여결손 `D = AUROC(12) − AUROC(5전극)` | `D ≤ 0` (실험20 과 같은 방향) |
| **P-4★★** | **동작점 안정성** — 앙상블 임계값이 목표 민감도에 더 가깝게 착지하나 | `E_단일 − E_앙상블 > 0` (레코드 부트스트랩 짝지은 CI) |

## 실험20b 에서 새로 들어온 규약

- **시드 3개 → 5개.** t 배수가 4.303 → 2.776 으로 내려간다. 실험20b 에서 임계값 기반
  지표가 시드 3개로는 검정력이 없다는 게 드러났다. 학습 **10회**(2구성 × 5시드).
- **오즈비·비율 CI 는 로그 스케일**에서 잡고 지수화한다(실험20b 에서 오즈비 CI 가
  음수로 나오는 오류를 냈다).
- **시드 t-CI 와 부트스트랩 CI 를 둘 다** 내고 **넓은 쪽**으로 판정한다.
- **임계값은 앙상블 위에서**도 함께 잡아 두 방식을 비교한다(P-4).

## 설계 안전장치 (실험20 에서 값을 치른 것들)

1. **헤더만 먼저** 받아 SNOMED 전수 열거 → MI 가 충분한지 확인하고 **신호는 그 뒤에**.
   부정맥 중심 데이터베이스라 **MI 가 거의 없을 수 있다** — 그때는 GB 를 받기 전에 멈춘다.
2. 필요한 파일을 **헤더에서** 읽는다(확장자 추측 금지). 잘린 다운로드는 예상 바이트로 잡는다.
3. **MI 전량 + 대조군 층화 표본**만 받는다. 45,152 전량은 캐시가 2GB 를 넘는데
   AUROC·민감도·특이도는 유병률과 무관하므로 필요 없다. **PPV 는 인구조사의 진짜
   유병률로 재구성**한다(표본 유병률로 계산하면 틀린다).
4. 캐니어리 1건 → 유도 순서 항등식 → 대역 정합 사다리 순으로 관문을 통과해야 학습으로 간다.

## 하지 않는 것

- 부위 국소화(SNOMED 에 없다) · 임계값 외부 재조정 · 백본 변경


In [ ]:
# CELL 0 — 공용 사전점검
class LabelVocabError(ValueError):
    pass

# ── pipelines/ecg_preflight.py 인라인 (원본·테스트는 repo)
def assert_label_vocab(requested, available, kind="label", counts=None, min_count=1):
    """요청한 이름이 실제 어휘에 **전부** 있는지 확인한다. 하나라도 없으면 예외.

    0건은 "데이터에 그 소견이 없다"가 아니라 **대개 이름을 잘못 골랐다**는 뜻이다.
    그 둘을 구별하려고 어휘 자체를 대조한다.

    requested : 쓰려는 이름들
    available : 데이터에서 실제로 관측된 이름 집합
    counts    : {이름: 건수} (있으면 min_count 미만도 함께 보고)
    """
    requested, available = list(requested), set(available)
    unknown = [r for r in requested if r not in available]
    if unknown:
        raise LabelVocabError(
            f"{kind} 어휘에 없는 이름 {unknown}.\n"
            f"  → 0건이 나온 이유는 '데이터에 없어서'가 아니라 **이름이 틀려서**다.\n"
            f"  실제 어휘({len(available)}개): {sorted(available)}"
        )
    thin = []
    if counts:
        thin = [(r, counts.get(r, 0)) for r in requested if counts.get(r, 0) < min_count]
    return {"ok": True, "n_requested": len(requested), "thin": thin}

def decide(lo, hi, thr, direction):
    """사전등록 관문의 **유일한** 계약: 지지(True) / 기각(False) / 미결(None).

    CI 가 임계값을 걸치면 **기각이 아니라 미결**이다. 검정력 부족을 반증으로
    위장하지 않기 위해서다. 점추정 2분 채점은 금지한다(실험13b·14 에서 그 실수를 했다).
    """
    if direction not in (">", "<"):
        raise ValueError("direction 은 '>' 또는 '<'")
    if direction == ">":
        if lo > thr:
            return True
        if hi < thr:
            return False
    else:
        if hi < thr:
            return True
        if lo > thr:
            return False
    return None

MARK = {True: "✅ 지지", False: "❌ 기각", None: "⚠️ 미결"}

def assert_arm_shape(arm, expected_rows, name="arm"):
    """저장된 arm 의 행 수가 **겹 크기**인지 확인한다.

    MedKOSRun.save_arm 은 그 겹의 예측만 저장한다(전체 길이가 아니다).
    겹 순서는 `np.where(CV == k)[0]` 의 오름차순이므로,
      OOF[np.where(CV == k)[0]] = load_arm(...)      ← 이렇게 **넣는다**
      load_arm(...)[전역인덱스]                       ← 이렇게 자르면 IndexError
    실험15d G0 에서 이 혼동으로 터졌다.
    """
    n = arm.shape[0]
    if n != expected_rows:
        raise ValueError(
            f"{name} 행 수 {n} != 기대 {expected_rows}.\n"
            "  → arm 은 **겹 크기**로 저장된다. 전역 인덱스로 자르지 말고 "
            "OOF[np.where(CV==k)[0]] = arm 형태로 넣을 것."
        )
    return {"ok": True, "rows": n}

FRONTAL_IDENTITIES = (
    ("III", 2, lambda I, II: II - I),                 # 아인트호벤
    ("aVR", 3, lambda I, II: -(I + II) / 2.0),        # 골드버거
    ("aVL", 4, lambda I, II: I - II / 2.0),
    ("aVF", 5, lambda I, II: II - I / 2.0),
)

def assert_lead_order(X, tol=0.02, sample=200, seed=0):
    """12유도 캐시의 **채널 순서**를 신호 자체로 검증한다.

    헤더의 유도 이름을 믿지 말고 아인트호벤·골드버거 항등식으로 확인한다:
        III = II − I,  aVR = −(I+II)/2,  aVL = I − II/2,  aVF = II − I/2
    넷이 모두 맞으면 0..5 = I,II,III,aVR,aVL,aVF 이고 표준 순서상 6..11 = V1..V6 이다.
    → `{I,II}` = [0,1], `{II,V1}` = [1,6] 을 쓸 근거가 생긴다.

    유도 순서를 틀리면 **예외 없이 조용히 다른 실험**이 된다. 그래서 잰다.
    ※ 원신호(mV) 전제 — 채널별로 정규화한 배열에는 쓸 수 없다.
    """
    import numpy as np
    if X.ndim != 3 or X.shape[2] != 12:
        raise ValueError(f"X 는 (n, t, 12) 여야 한다 — 받은 모양 {X.shape}")
    rs = np.random.RandomState(seed)
    idx = rs.choice(len(X), size=min(sample, len(X)), replace=False)
    S = X[idx].astype("float64")
    I, II = S[:, :, 0], S[:, :, 1]
    report, bad = {}, []
    for name, j, f in FRONTAL_IDENTITIES:
        want = f(I, II)
        scale = np.abs(want).mean() + 1e-9
        err = float(np.abs(S[:, :, j] - want).mean() / scale)
        report[name] = err
        if err > tol:
            bad.append(f"{name}(ch{j}) 상대오차 {err:.3f}")
    if bad:
        raise ValueError(
            "유도 순서가 표준(I,II,III,aVR,aVL,aVF,V1..V6)이 아니다: " + ", ".join(bad) + "\n"
            "  → 항등식이 깨졌다는 것은 채널 배치가 다르거나 채널별 정규화가 걸렸다는 뜻이다.\n"
            "  마스크 인덱스([0,1] 사지 · [1,6] II+V1)를 그대로 쓰면 조용히 다른 실험이 된다."
        )
    return {"ok": True, "n_checked": len(idx), "rel_err": report}

def boot_indices(n, B, seed):
    """부트스트랩 인덱스를 **생성기로** 돌려준다.

    미리 리스트로 만들면 B=4000, n=16k 에서 520MB 다. 같은 시드로 매번 다시 돌리면
    메모리 0 이면서 **여러 군이 같은 재표본 축을 공유**한다(짝지은 비교의 전제).
    """
    import numpy as np
    rs = np.random.RandomState(seed)
    for _ in range(B):
        yield rs.randint(0, n, n)

print("사전점검 적재: assert_label_vocab · decide · assert_arm_shape · "
      "assert_lead_order · boot_indices")

In [ ]:
# CELL 1 — 설정
!pip -q install wfdb

import os, sys, json, time, re, ast, subprocess, numpy as np
try:
    from google.colab import drive; drive.mount("/content/drive", force_remount=False)
    DRIVE_ROOT = "/content/drive/MyDrive"
except Exception as e:
    print("⚠️ Colab 아님:", e); DRIVE_ROOT = "/content"
PROJECT = os.path.join(DRIVE_ROOT, "MedKOS", "ecg-model")
sys.path.insert(0, os.path.join(PROJECT, "lib"))
from medkos_run import MedKOSRun

# ★★ 실험15~22 와 한 글자도 달라선 안 되는 블록
K_FOLD, EPOCHS, SEED0, NMIN = 5, 20, 20260801, 50
SITE_CANDIDATES = ["IMI", "ILMI", "IPMI", "IPLMI", "ASMI", "AMI", "ALMI", "LMI", "PMI"]
LEADS = {"I+II+V2+V5": [0, 1, 7, 10], "12": list(range(12))}
# ★★ 여기까지

DEPLOY, REF = "I+II+V2+V5", "12"
SEEDS = [0, 1, 2, 3, 4]          # ★ 3 → 5. 실험20b: 임계값 지표는 3시드로 검정력이 없다
FS_OUT, SEC, SKIP = 100, 10, 0   # Chapman/Ningbo 는 10초 고정이라 스킵 없이 전체를 쓴다
SENS_TARGET = 0.90
DROP_THR, D_THR, LRNEG_THR = 0.10, 0.0, 0.20
# ★ P-4 를 "단일 SD / 앙상블 SD >= 1.5" 로 재려던 설계를 **폐기했다**.
#   leave-one-out 앙상블 5개는 서로 4/5 를 공유해 SD 가 구조적으로 1/4 이 된다 —
#   효과가 0 이어도 비가 정확히 4.00 이 나온다(픽스처 ⑤ 가 수치로 잡았다).
#   대신 **목표 민감도에서 얼마나 벗어났나**를 레코드 부트스트랩으로 짝지어 비교한다.
GMIN_MI, N_CTRL = 200, 6000      # MI 최소 건수 · 대조군 표본 크기(층화)
BOOT = 2000
CN_URL = "https://physionet.org/files/ecg-arrhythmia/1.0.0"
PTBXL_URL = "https://physionet.org/files/ptb-xl/1.0.3"

# ── MI 코드 선정 규칙을 **미리** 선언한다(결과를 보고 고르면 사후 조정이다)
MI_NAME_RE = re.compile(r"infarct", re.I)          # 이름에 'infarct' 가 들어가면 후보
MI_EXCLUDE_RE = re.compile(r"no |absent|rule.?out", re.I)   # 부정 표현은 뺀다

CONFIG = dict(exp="exp21_chapman_ningbo", quest="ailab-2026-0015",
              parent_exp=["exp20_ptbdb", "exp20b_patient", "exp19_two_stage"],
              purpose=("두 번째 외부 병원(중국 2기관·다른 기기·부정맥 코호트)에서 "
                       "실험20 의 세 발견을 복제 검정한다. SNOMED 라벨이라 MI-any 한 축"),
              dataset="PhysioNet ecg-arrhythmia 1.0.0 (Chapman-Shaoxing + Ningbo)",
              change_one_thing="백본·전처리 규격 동일. 평가 데이터와 라벨 층위(MI-any)만 다름",
              configs=list(LEADS), deploy=DEPLOY, seeds=SEEDS,
              seeds_note="실험20b 에서 3시드는 t 배수 4.303 이라 임계값 지표에 검정력이 없었다",
              mi_rule=f"이름 정규식 {MI_NAME_RE.pattern} 채택 · {MI_EXCLUDE_RE.pattern} 제외",
              sampling=(f"MI 전량 + 대조군 층화 표본 {N_CTRL}. AUROC·민감도·특이도는 "
                        "유병률 무관이고 PPV 는 **인구조사의 진짜 유병률**로 재구성한다"),
              ci_rule=("**유계 지표(민감도·특이도)는 logit**, **비율·오즈비는 log** 스케일에서 "
                       "CI 를 잡고 되돌린다(실험20b 에서 특이도 CI 가 -0.183, 오즈비 CI 가 "
                       "-0.47 로 나오는 오류를 냈다). 시드 t-CI 와 부트스트랩 CI 를 둘 다 "
                       "내고 넓은 쪽으로 판정"),
              predictions={
                  "G0-a": f"SNOMED 미매핑 0 · MI 레코드 >= {GMIN_MI} 아니면 중단",
                  "P-1": f"MI-any 외부 AUROC 낙폭 < {DROP_THR}",
                  "P-2": f"동결 동작점의 외부 LR- <= {LRNEG_THR} (rule-out 이 살아남나)",
                  "P-3": f"외부 잔여결손 D = AUROC(12)-AUROC(5전극) <= {D_THR} (실험20 복제)",
                  "P-4": ("앙상블 임계값이 목표 민감도에 더 가깝게 착지한다 — "
                          "E_단일(시드 평균 |Se−목표|) − E_앙상블 > 0, 레코드 부트스트랩 짝지은 CI")},
              caveat=("부위 국소화 없음(SNOMED 한계) · 부정맥 중심 코호트라 MI 가 적을 수 있다 · "
                      "내부는 5겹 OOF·외부는 전량 학습이라 낙폭은 과소추정"),
              k_fold=K_FOLD, epochs=EPOCHS, seed0=SEED0, boot=BOOT)
np.random.seed(SEED0)
try:
    import matplotlib, matplotlib.font_manager as fm
    if not any("Nanum" in f.name for f in fm.fontManager.ttflist):
        subprocess.run(["apt-get", "-qq", "install", "-y", "fonts-nanum"], capture_output=True)
        fm.fontManager.addfont("/usr/share/fonts/truetype/nanum/NanumGothic.ttf")
    matplotlib.rc("font", family="NanumGothic")
    matplotlib.rcParams["axes.unicode_minus"] = False
except Exception as e:
    print("한글 폰트 설정 생략:", e)

run = MedKOSRun("exp21_chapman", CONFIG, project=PROJECT)

def t_ci(v, conf=.95):
    from scipy import stats
    v = np.asarray([x for x in v if np.isfinite(x)], float); n = len(v)
    m = float(v.mean()) if n else float("nan")
    if n < 2:
        return m, np.nan, np.nan
    h = float(stats.t.ppf(.5 + conf / 2, n - 1) * v.std(ddof=1) / np.sqrt(n))
    return m, m - h, m + h

def t_ci_logit(v, conf=.95):
    """★ **유계 지표(0~1)** 는 logit 스케일에서 CI 를 잡는다.
    실험20b 에서 특이도 t-CI 가 [-0.183,+0.575] 로 나왔다 — 특이도는 음수가 될 수 없다."""
    v = np.clip(np.asarray([x for x in v if np.isfinite(x)], float), 1e-6, 1 - 1e-6)
    if len(v) < 2:
        return (float(v.mean()) if len(v) else np.nan), np.nan, np.nan
    z = np.log(v / (1 - v))
    m, lo, hi = t_ci(z, conf)
    f = lambda x: float(1 / (1 + np.exp(-x)))
    return f(m), f(lo), f(hi)

def t_ci_log(v, conf=.95):
    """★ 비율·오즈비는 **로그 스케일**에서 CI 를 잡고 지수화한다.
    실험20b 에서 오즈비에 그냥 t-CI 를 씌워 [-0.47,+3.63] 이 나왔다 — 음수 오즈비는 없다."""
    v = np.asarray([x for x in v if np.isfinite(x) and x > 0], float)
    if len(v) < 2:
        return (float(v.mean()) if len(v) else np.nan), np.nan, np.nan
    m, lo, hi = t_ci(np.log(v), conf)
    return float(np.exp(m)), float(np.exp(lo)), float(np.exp(hi))


In [ ]:
# CELL 2 — 【G0-a】 헤더만 받아 SNOMED 전수 열거 → MI 코드 선정 → 인구조사
#   ★ 부정맥 중심 데이터베이스다. **MI 가 거의 없을 수 있다.**
#     신호(수 GB)를 받기 전에 여기서 끝낼 수 있어야 한다(실험20 의 교훈).
import pandas as pd, wfdb

CN = "/content/cn"; os.makedirs(CN, exist_ok=True)
HDR_CACHE = run.data("chapman_ningbo_headers_v1.json")

def wget(url, out, quiet=True):
    subprocess.run(["wget"] + (["-q"] if quiet else []) + ["-O", out, url], check=False)
    return os.path.exists(out) and os.path.getsize(out) > 0

if os.path.exists(HDR_CACHE):
    H = pd.DataFrame(json.load(open(HDR_CACHE, encoding="utf-8")))
    run.log(f"헤더 캐시 적중 {HDR_CACHE} · {len(H)}건")
else:
    run.log("헤더(.hea)만 재귀 수신 — 신호는 아직 안 받는다")
    t0 = time.time()
    subprocess.run(["wget", "-q", "-r", "-N", "-c", "-np", "-nH", "--cut-dirs=3",
                    "-A", "*.hea", "-P", CN,
                    f"{CN_URL}/WFDBRecords/"], check=False)
    heas = []
    for root, _, files in os.walk(CN):
        heas += [os.path.join(root, f) for f in files if f.endswith(".hea")]
    run.log(f"헤더 {len(heas)}개 · {time.time()-t0:.0f}s")
    if len(heas) < 1000:
        raise RuntimeError(
            f"헤더가 {len(heas)}개뿐이다 — 재귀 수신이 실패했다.\n"
            f"  → {CN_URL}/WFDBRecords/ 구조를 확인할 것. 신호 다운로드로 넘어가지 않는다.")
    rows = []
    for p in heas:
        h = wfdb.rdheader(p[:-4])
        d = {"rec": os.path.relpath(p, CN)[:-4], "fs": h.fs, "sig_len": h.sig_len,
             "sig_name": ",".join(s.lower() for s in h.sig_name), "dx": "", "age": "", "sex": ""}
        for c in (h.comments or []):
            if ":" in c:
                k, v = c.split(":", 1)
                k = k.strip().lower()
                if k in ("dx", "age", "sex"):
                    d[k] = v.strip()
        rows.append(d)
    H = pd.DataFrame(rows)
    json.dump(H.to_dict("records"), open(HDR_CACHE, "w", encoding="utf-8"), ensure_ascii=False)
    run.log(f"헤더 캐시 생성 {HDR_CACHE} · {len(H)}건")

# ── SNOMED 코드 → 이름. 데이터베이스가 제공하는 표를 쓴다(우리가 외우지 않는다)
CN_MAP = os.path.join(CN, "ConditionNames_SNOMED-CT.csv")
if not os.path.exists(CN_MAP):
    wget(f"{CN_URL}/ConditionNames_SNOMED-CT.csv", CN_MAP)
if not (os.path.exists(CN_MAP) and os.path.getsize(CN_MAP) > 0):
    raise RuntimeError(
        f"SNOMED 이름표를 못 받았다: {CN_URL}/ConditionNames_SNOMED-CT.csv\n"
        "  → 코드 번호를 **추측해서 MI 를 고르지 않는다.** 이름표 없이는 진행 불가.")
NM = pd.read_csv(CN_MAP)
code_col = next(c for c in NM.columns if "code" in c.lower())
name_col = next(c for c in NM.columns if "name" in c.lower() or "full" in c.lower())
NAME = {str(r[code_col]).strip(): str(r[name_col]).strip() for _, r in NM.iterrows()}
run.log(f"SNOMED 이름표 {len(NAME)}개 · 열 {code_col} / {name_col}")

# ── 전수 열거. 이름표에 없는 코드가 하나라도 있으면 멈춘다
from collections import Counter
cnt = Counter()
for v in H.dx.fillna(""):
    for c in [x.strip() for x in str(v).split(",") if x.strip()]:
        cnt[c] += 1
unknown = sorted(c for c in cnt if c not in NAME)
run.log(f"\n관측된 SNOMED 코드 {len(cnt)}종 · 이름표에 없는 것 {len(unknown)}종")
if unknown:
    run.log("  이름표에 없는 코드(상위 10): "
            + ", ".join(f"{c}({cnt[c]})" for c in sorted(unknown, key=lambda x: -cnt[x])[:10]))
    raise LabelVocabError(
        f"SNOMED 이름표에 없는 코드 {len(unknown)}종. **추측해서 넘어가지 않는다.**\n"
        "  → 이름표 버전을 확인하거나 해당 코드를 명시적으로 무시 목록에 넣고 다시 돌린다.")

# ── MI 코드 선정: **사전 선언한 이름 정규식**으로만 고른다
mi_codes = sorted([c for c in cnt if MI_NAME_RE.search(NAME[c])
                   and not MI_EXCLUDE_RE.search(NAME[c])], key=lambda x: -cnt[x])
excluded = [c for c in cnt if MI_NAME_RE.search(NAME[c]) and MI_EXCLUDE_RE.search(NAME[c])]
run.log("\n【MI 코드 선정】 사전 선언한 규칙만 적용 — 결과를 보고 고르지 않는다")
for c in mi_codes:
    run.log(f"  채택 {c:<16}{cnt[c]:>7}건  {NAME[c]}")
for c in excluded:
    run.log(f"  제외 {c:<16}{cnt[c]:>7}건  {NAME[c]}  (부정 표현)")
if not mi_codes:
    raise RuntimeError(
        "이름에 'infarct' 가 들어가는 SNOMED 코드가 하나도 없다 — 이 데이터베이스로는\n"
        "  MI 외부검증을 할 수 없다. **신호를 받지 않고 여기서 끝낸다.**\n"
        f"  (관측된 코드 상위 15: "
        + ", ".join(f"{NAME[c]}({cnt[c]})" for c, _ in cnt.most_common(15)) + ")")

MI = np.array([any(c in mi_codes for c in str(v).split(",")) for v in H.dx.fillna("")])
H["mi"] = MI
n_mi = int(MI.sum())
prev_true = float(MI.mean())
run.log(f"\n【인구조사】 전체 {len(H):,}건 · MI {n_mi:,}건 · **진짜 유병률 {prev_true:.3%}**")
if n_mi < GMIN_MI:
    raise RuntimeError(
        f"MI 레코드가 {n_mi}건뿐이다(문턱 {GMIN_MI}). 부정맥 중심 코호트라 예상했던 위험이다.\n"
        "  → **신호를 받지 않고 중단한다.** 실험21 은 '이 데이터로는 MI 외부검증 불가' 로\n"
        "     종결하고 실험24·23 으로 넘어간다. 이것도 결과다(음성 결과를 기록한다).")
run.log(f"  ✅ MI {n_mi}건 >= {GMIN_MI} → 신호 수신으로 진행")

# ── 표본 선정: MI 전량 + 대조군 층화(디렉터리 접두 = 배치/기관 대리 변수)
H["stratum"] = H.rec.str.split("/").str[0]
rs = np.random.RandomState(SEED0)
ctrl = H[~H.mi]
take = min(N_CTRL, len(ctrl))
frac = take / len(ctrl)
sel_ctrl = (ctrl.groupby("stratum", group_keys=False)
            .apply(lambda g: g.sample(max(1, int(round(len(g) * frac))), random_state=SEED0)))
SEL = pd.concat([H[H.mi], sel_ctrl]).drop_duplicates("rec").reset_index(drop=True)
run.log(f"  표본: MI {int(SEL.mi.sum()):,} + 대조 {int((~SEL.mi).sum()):,} = {len(SEL):,}건 "
        f"({SEL.stratum.nunique()}개 층에서 층화)")
run.log(f"  ※ 표본 유병률 {SEL.mi.mean():.1%} 는 **인위적**이다. PPV 는 진짜 유병률 "
        f"{prev_true:.3%} 로 재구성한다")
CONFIG["prevalence_true"] = prev_true
CONFIG["mi_codes"] = {c: NAME[c] for c in mi_codes}
run.save_json("config", CONFIG)


In [ ]:
# CELL 3 — 신호 수신 + 전처리 + 【G0-b】 유도 순서·대역 정합
from scipy.signal import resample_poly, butter, filtfilt
from collections import Counter

SIG_CACHE = run.data("chapman_ningbo_12lead_100hz_v1.npz")
ORDER = ["i", "ii", "iii", "avr", "avl", "avf", "v1", "v2", "v3", "v4", "v5", "v6"]
FMT_BYTES = {"16": 2, "61": 2, "160": 2, "80": 1}

def rec_files(rec):
    """필요한 파일을 **헤더에서** 읽는다(확장자 추측 금지 — 실험20 의 .xyz 사고)."""
    h = wfdb.rdheader(os.path.join(CN, rec))
    names = [s.lower() for s in h.sig_name]
    miss = [nm for nm in ORDER if nm not in names]
    if miss:
        raise RuntimeError(f"{rec}: 12유도 중 {miss} 가 없다 · {names}")
    d = os.path.dirname(rec)
    out = []
    for f in sorted({h.file_name[names.index(nm)] for nm in ORDER}):
        js = [j for j, x in enumerate(h.file_name) if x == f]
        fmts = {str(h.fmt[j]) for j in js}
        nb_ = FMT_BYTES.get(fmts.pop()) if len(fmts) == 1 else None
        out.append(((os.path.join(d, f) if d else f),
                    int(h.sig_len) * len(js) * nb_ if nb_ else None))
    return out

def fetch(rel, exp=None):
    """잘린 파일을 남기지 않는다(실험20 에서 2건이 잘린 채 굳었다)."""
    p = os.path.join(CN, rel)
    for _ in (0, 1):
        if os.path.exists(p):
            n = os.path.getsize(p)
            if n > 512 and (exp is None or n >= exp):
                return True
            os.remove(p)
        os.makedirs(os.path.dirname(p), exist_ok=True)
        subprocess.run(["wget", "-q", "-O", p, f"{CN_URL}/WFDBRecords/{rel}"], check=False)
    n = os.path.getsize(p) if os.path.exists(p) else 0
    if n <= 512 or (exp is not None and n < exp):
        if os.path.exists(p):
            os.remove(p)
        return False
    return True

def read12(rec):
    h = wfdb.rdheader(os.path.join(CN, rec))
    names = [s.lower() for s in h.sig_name]
    ch = sorted({names.index(nm) for nm in ORDER})
    sig, f = wfdb.rdsamp(os.path.join(CN, rec), channels=ch)
    got = [s.lower() for s in f["sig_name"]]
    return sig[:, [got.index(nm) for nm in ORDER]], float(f["fs"])

if not os.path.exists(SIG_CACHE):
    # 【G0-b0】 캐니어리 1건
    c0_ = SEL.rec.iloc[0]
    rels = rec_files(c0_)
    got = [fetch(r_, e_) for r_, e_ in rels]
    try:
        s0, fs0 = read12(c0_)
    except Exception as e:
        raise RuntimeError(f"캐니어리 {c0_} 실패 {type(e).__name__}: {str(e)[:160]}\n"
                           f"  받은 파일: {list(zip(rels, got))}\n"
                           "  → 전량 수신으로 넘어가지 않는다.")
    run.log(f"【G0-b0】 캐니어리 {c0_} ✅ {s0.shape} @ {fs0:.0f}Hz · 파일 {[r for r, _ in rels]}")

    run.log(f"신호 {len(SEL):,}건 수신 (표본만 — 전량 45k 는 안 받는다)")
    t0, miss = time.time(), []
    for i, r in enumerate(SEL.rec):
        for rel, exp in rec_files(r):
            if not fetch(rel, exp):
                miss.append(rel)
        if (i + 1) % 500 == 0:
            run.log(f"  {i+1}/{len(SEL)} · {time.time()-t0:.0f}s"
                    + (f" · 실패 {len(miss)}" if miss else ""))
    run.log(f"수신 {time.time()-t0:.0f}s · 실패 {len(miss)}건 · 전처리 시작")

    n_out = SEC * FS_OUT
    X, keep, bad = [], [], []
    for r in SEL.rec:
        try:
            seg, fs = read12(r)
            a = int(SKIP * fs); b = a + int(SEC * fs)
            if seg.shape[0] < b:
                bad.append((r, "짧음")); continue
            seg = seg[a:b].astype("float64")
            if not np.isfinite(seg).all():
                bad.append((r, "NaN")); continue
            g = int(round(fs / FS_OUT))
            seg = resample_poly(seg, 1, g, axis=0) if g > 1 else seg
            if seg.shape[0] != n_out:
                bad.append((r, f"길이 {seg.shape[0]}")); continue
            X.append(seg.astype("float32")); keep.append(r)
        except Exception as e:
            bad.append((r, f"{type(e).__name__}: {str(e)[:40]}"))
    if not X:
        c_ = Counter(w for _, w in bad)
        raise RuntimeError(f"전처리 0건 (실패 {len(bad)}/{len(SEL)}). 사유:\n"
                           + "\n".join(f"    {n:>5}건  {w}" for w, n in c_.most_common(8)))
    X = np.stack(X)
    run.log(f"전처리 완료 X{X.shape} · 실패 {len(bad)}건")
    for w, n in Counter(w for _, w in bad).most_common(5):
        run.log(f"    실패 사유 {n:>5}건  {w}")
    np.savez_compressed(SIG_CACHE, X=X, recs=np.array(keep))
    del X
z = np.load(SIG_CACHE, allow_pickle=True)
XE, RKEEP = z["X"], [str(x) for x in z["recs"]]
YE = SEL.set_index("rec").loc[RKEEP, "mi"].values.astype(bool)
run.log(f"외부 캐시 {XE.shape} · MI {int(YE.sum()):,} / {len(YE):,}")

chk = assert_lead_order(XE)
run.log("【G0-b1】 유도 순서 ✅ " + " · ".join(f"{k} {v:.4f}" for k, v in chk["rel_err"].items()))

# 【G0-b2】 대역 정합 — 실험20 과 **같은 사다리**를 쓴다(새 자유도를 만들지 않는다)
PX = run.data("ptbxl_12lead_all.npz")
if not os.path.exists(PX):
    raise RuntimeError(f"PTB-XL 전량 캐시가 없습니다: {PX}")
zx = np.load(PX, allow_pickle=True)
XI, FOLD10, EID = zx["X"], zx["fold"], zx["eid"]
CV = (FOLD10 - 1) % K_FOLD
rs_ = np.random.RandomState(SEED0)
sxl = rs_.choice(len(XI), 500, replace=False)
sdb = rs_.choice(len(XE), min(500, len(XE)), replace=False)

def decomp(S):
    return dict(within=np.median(S.std(axis=1), axis=0),
                offset=S.mean(axis=1).std(axis=0))
def demean(X):
    return (X - X.mean(axis=1, keepdims=True)).astype("float32")
def hp(X, fc, order=3):
    b, a = butter(order, fc / (FS_OUT / 2), btype="high")
    return filtfilt(b, a, X, axis=1).astype("float32")
LADDER = [("원본", lambda X: X), ("레코드별 평균 제거", demean),
          ("0.5Hz 0위상 고역통과", lambda X: hp(demean(X), 0.5))]
OFF_THR = 0.25
CONFIG["band_ladder"] = [n for n, _ in LADDER]; CONFIG["off_thr"] = OFF_THR

di = decomp(XI[sxl])
run.log("\n【G0-b2】 대역 정합 (실험20 과 동일 사다리 · 통과하는 첫 단계만)")
BAND, SCALE = None, None
for name, fn in LADDER:
    de = decomp(fn(XE[sdb])); dl = decomp(fn(XI[sxl]))
    ratio = float(np.nanmedian(de["within"] / di["within"]))
    r_off = float(np.median(de["offset"]) / np.median(di["within"]))
    noop = float(np.nanmax(np.abs(dl["within"] - di["within"]) / di["within"]))
    ok_ = (0.5 <= ratio <= 2.0) and (r_off <= OFF_THR) and (noop < 0.10)
    run.log(f"  · {name:<20} 진폭비 {ratio:>5.2f} · 오프셋 {r_off:>5.2f} · "
            f"PTB-XL 변화 {noop:>6.1%} {'✅' if ok_ else '❌'}")
    if ok_:
        BAND, SCALE = (name, fn), ratio
        break
if BAND is None:
    raise RuntimeError("사전 선언한 사다리로 진폭이 안 맞는다 — 사다리를 지금 늘리지 않는다")
run.log(f"  → 채택: **{BAND[0]}** (비 {SCALE:.2f})")
if BAND[0] != "원본":
    XE = BAND[1](XE)
    run.log("  【G0-b1 재확인】 " + " · ".join(
        f"{k} {v:.4f}" for k, v in assert_lead_order(XE)["rel_err"].items()))
CONFIG["band_applied"] = BAND[0]; run.save_json("config", CONFIG)


In [ ]:
# CELL 4 — PTB-XL 전량 **MI-any** 학습 (2구성 × 5시드 = 10회)
import tensorflow as tf
from tensorflow.keras import layers, models

px_csv, px_scp = "/content/ptbxl/ptbxl_database.csv", "/content/ptbxl/scp_statements.csv"
os.makedirs("/content/ptbxl", exist_ok=True)
for f, u in ((px_csv, f"{PTBXL_URL}/ptbxl_database.csv"),
             (px_scp, f"{PTBXL_URL}/scp_statements.csv")):
    if not os.path.exists(f):
        subprocess.run(["wget", "-q", "-O", f, u], check=True)
dfa = pd.read_csv(px_csv, index_col="ecg_id").loc[EID]
scp = pd.read_csv(px_scp, index_col=0)

# ★ MI 코드는 **PTB-XL 공식 표**에서 가져온다(우리가 목록을 손으로 쓰지 않는다)
MI_SCP = sorted(scp[(scp.diagnostic == 1) & (scp.diagnostic_class == "MI")].index.astype(str))
dfa["codes"] = dfa.scp_codes.apply(lambda s: sorted(ast.literal_eval(s).keys()))
vocab = {c for cs in dfa.codes for c in cs}
assert_label_vocab(MI_SCP, vocab, kind="PTB-XL MI 코드")
YI = np.array([any(c in MI_SCP for c in cs) for cs in dfa.codes], "float32")
run.log(f"내부 MI-any 라벨 — 코드 {MI_SCP}")
run.log(f"  양성 {int(YI.sum()):,} / {len(YI):,} = {YI.mean():.1%}")

MASKS = {c: np.zeros(12, "float32") for c in LEADS}
for c, idx in LEADS.items():
    MASKS[c][idx] = 1.0

def build_head(seed):
    """★ 실험15~20 과 완전히 동일. 출력만 1개(MI-any 이진)."""
    tf.keras.utils.set_random_seed(seed)
    si_ = layers.Input((XI.shape[1], 12))
    x = si_
    for f, k in ((32, 9), (64, 7), (128, 5), (128, 3)):
        x = layers.Conv1D(f, k, padding="same", activation="relu")(x)
        x = layers.BatchNormalization()(x)
        x = layers.MaxPooling1D(2)(x)
    h = layers.Dense(64, activation="relu")(layers.GlobalAveragePooling1D()(x))
    h = layers.Dropout(0.3)(h); h = layers.Dense(64, activation="relu")(h)
    m = models.Model(si_, layers.Dense(1, activation="sigmoid")(h))
    m.compile(optimizer=tf.keras.optimizers.Adam(1e-3, clipnorm=1.0),
              loss="binary_crossentropy")
    return m

rs2 = np.random.RandomState(SEED0); order = rs2.permutation(len(XI))
n_val = max(int(len(order) * 0.12), 200)
VA, TR = order[:n_val], order[n_val:]
run.log(f"전량 학습 — 학습 {len(TR):,} · 검증 {len(VA):,}")

PE, PI = {c: {} for c in LEADS}, {c: {} for c in LEADS}   # 외부 · 내부(전량모델의 검증셋)
t0, done = time.time(), 0
for c in LEADS:
    mk = MASKS[c]
    for sd in SEEDS:
        ae, ai = run.load_arm(f"ext_{c}_s{sd}"), run.load_arm(f"inval_{c}_s{sd}")
        if ae is None or ai is None:
            m = build_head(SEED0 + 21 + sd)
            m.fit(XI[TR] * mk, YI[TR], validation_data=(XI[VA] * mk, YI[VA]),
                  epochs=EPOCHS, batch_size=128, verbose=0)
            ae = m.predict(XE * mk, batch_size=512, verbose=0)
            ai = m.predict(XI[VA] * mk, batch_size=512, verbose=0)
            run.save_arm(f"ext_{c}_s{sd}", ae); run.save_arm(f"inval_{c}_s{sd}", ai)
            if sd == SEEDS[0]:
                run.save_model(m, f"full_{c}")
            tf.keras.backend.clear_session(); done += 1
            run.log(f"  {c:<12} 시드{sd} 완료 ({done} · {time.time()-t0:.0f}s)")
        assert_arm_shape(ae, len(XE), name=f"ext_{c}_s{sd}")
        assert_arm_shape(ai, len(VA), name=f"inval_{c}_s{sd}")
        PE[c][sd] = ae[:, 0]; PI[c][sd] = ai[:, 0]
YIV = YI[VA].astype(bool)
run.log(f"총 {time.time()-t0:.0f}s · 이번 세션 학습 {done}회")


In [ ]:
# CELL 5 — 【P-1·P-3】 외부 AUROC 낙폭 · 잔여결손 D (실험20 복제)
from sklearn.metrics import roc_auc_score

def auc(y, s):
    return float(roc_auc_score(y, s)) if y.any() and (~y).any() else np.nan

run.log("\n" + "=" * 110)
run.log("【P-1·P-3】 MI-any — 내부(전량모델의 내부 검증셋) vs 외부(Chapman/Ningbo)")
run.log("=" * 110)
INT = {c: [auc(YIV, PI[c][sd]) for sd in SEEDS] for c in LEADS}
EXT = {c: [auc(YE, PE[c][sd]) for sd in SEEDS] for c in LEADS}
run.log(f"  {'구성':<14}{'내부':>10}{'외부':>10}{'낙폭':>10}")
for c in LEADS:
    d = [INT[c][i] - EXT[c][i] for i in range(len(SEEDS))]
    m, lo, hi = t_ci(d)
    run.log(f"  {c:<14}{np.mean(INT[c]):>10.4f}{np.mean(EXT[c]):>10.4f}"
            f"{m:>+10.4f} [{lo:+.4f},{hi:+.4f}]")
DROP = [INT[DEPLOY][i] - EXT[DEPLOY][i] for i in range(len(SEEDS))]
DE = [EXT[REF][i] - EXT[DEPLOY][i] for i in range(len(SEEDS))]
DI = [INT[REF][i] - INT[DEPLOY][i] for i in range(len(SEEDS))]
md, ld, hd = t_ci(DE)
mi_, li_, hi_ = t_ci(DI)
run.log(f"\n  잔여결손 D = AUROC(12) − AUROC(5전극)")
run.log(f"    내부 {mi_:+.4f} [{li_:+.4f},{hi_:+.4f}] · 외부 {md:+.4f} [{ld:+.4f},{hd:+.4f}]")
run.log("    ★ 실험20(PTBDB)에서는 외부 D 가 **음수**였다(12유도가 더 무너짐). 복제되나?")


In [ ]:
# CELL 6 — 【P-2·P-4】 동결 동작점 · rule-out · 앙상블 안정성
run.log("\n" + "=" * 110)
run.log("【P-2·P-4】 내부에서 동결한 동작점을 외부에 그대로 적용")
run.log("=" * 110)

def thr_sens(score, pos, t=SENS_TARGET):
    p = score[pos]
    return float(np.quantile(p, 1.0 - t, method="lower")) if len(p) else -np.inf

def op(score_int, score_ext):
    """내부에서 임계값을 잡고 외부 성능을 잰다. 임계값 재조정 없음."""
    th = thr_sens(score_int, YIV)
    sp_i = float((score_int[~YIV] < th).mean())
    se = float((score_ext[YE] >= th).mean()); sp = float((score_ext[~YE] < th).mean())
    lrn = (1 - se) / max(sp, 1e-9); lrp = se / max(1 - sp, 1e-9)
    return dict(thr=th, spec_int=sp_i, se=se, sp=sp, lrn=lrn, lrp=lrp)

run.log(f"  {'방식':<18}{'임계값':>10}{'내부sp':>9}{'외부se':>9}{'외부sp':>9}{'LR+':>8}{'LR−':>8}")
SING = [op(PI[DEPLOY][sd], PE[DEPLOY][sd]) for sd in SEEDS]
for sd, o in zip(SEEDS, SING):
    run.log(f"  단일 시드{sd:<10}{o['thr']:>10.5f}{o['spec_int']:>9.3f}"
            f"{o['se']:>9.3f}{o['sp']:>9.3f}{o['lrp']:>8.2f}{o['lrn']:>8.3f}")
th_all = np.array([o["thr"] for o in SING])
run.log(f"  → 임계값 최대/최소 = {th_all.max()/max(th_all[th_all>0].min(),1e-12):.1f}배"
        "   (실험20b 에서 8.4배였다)")

# ── 앙상블: 시드 하나를 빼고 평균 → 같은 개수(5)의 독립 관측을 만든다
ENS = []
for sd in SEEDS:
    rest = [s for s in SEEDS if s != sd]
    ENS.append(op(np.mean([PI[DEPLOY][s] for s in rest], 0),
                  np.mean([PE[DEPLOY][s] for s in rest], 0)))
for sd, o in zip(SEEDS, ENS):
    run.log(f"  앙상블(−시드{sd})   {o['thr']:>10.5f}{o['spec_int']:>9.3f}"
            f"{o['se']:>9.3f}{o['sp']:>9.3f}{o['lrp']:>8.2f}{o['lrn']:>8.3f}")

# ── 【P-4】 앙상블이 동작점을 목표에 더 가깝게 착지시키나
#   ★ "단일 SD / LOO앙상블 SD" 로 재면 **안 된다.** LOO 앙상블 5개는 서로 4/5 를 공유해
#     효과가 0 이어도 SD 비가 정확히 4.00 이 나온다. 대신 **목표 이탈**을 레코드
#     부트스트랩으로 짝지어 비교한다(같은 재표본을 두 방식이 공유 → 짝지은 비교).
TH_S = np.array([o["thr"] for o in SING])
TH_E = thr_sens(np.mean([PI[DEPLOY][s_] for s_ in SEEDS], 0), YIV)
SC_S = np.stack([PE[DEPLOY][s_] for s_ in SEEDS])              # (시드, 레코드)
SC_E = SC_S.mean(0)

def dev(idx):
    """그 재표본에서 (단일 평균 이탈, 앙상블 이탈)."""
    y = YE[idx]
    if not y.any():
        return np.nan, np.nan
    es = float(np.mean([abs((SC_S[k][idx][y] >= TH_S[k]).mean() - SENS_TARGET)
                        for k in range(len(SEEDS))]))
    ee = float(abs((SC_E[idx][y] >= TH_E).mean() - SENS_TARGET))
    return es, ee

d_obs = dev(np.arange(len(YE)))
diffs = []
for ix in boot_indices(len(YE), BOOT, SEED0):
    a, b = dev(ix)
    if np.isfinite(a) and np.isfinite(b):
        diffs.append(a - b)
blo, bhi = np.percentile(diffs, [2.5, 97.5])
run.log(f"\n  【P-4】 목표 민감도 {SENS_TARGET} 에서의 이탈")
run.log(f"      단일 평균 |Se−목표| {d_obs[0]:.4f} · 앙상블 {d_obs[1]:.4f} · "
        f"차 {d_obs[0]-d_obs[1]:+.4f} [{blo:+.4f},{bhi:+.4f}] (레코드 부트스트랩)")
run.log(f"      앙상블 임계값 {TH_E:.5f} · 시드 임계값 {np.round(TH_S,5).tolist()}")
run.log(f"      참고(설명용, 채점 아님) 달성민감도 SD — 단일 "
        f"{np.std([o['se'] for o in SING], ddof=1):.4f} · "
        f"LOO앙상블 {np.std([o['se'] for o in ENS], ddof=1):.4f} "
        "← LOO 는 서로 겹쳐 SD 가 구조적으로 작다. **비율로 채점하지 않는다**")
P4_LO, P4_HI, P4_DIFF = blo, bhi, d_obs[0] - d_obs[1]

# ── PPV 는 **진짜 유병률**로 재구성한다(표본 유병률로 계산하면 틀린다)
pv = CONFIG["prevalence_true"]
ens_all = op(np.mean([PI[DEPLOY][s] for s in SEEDS], 0),
             np.mean([PE[DEPLOY][s] for s in SEEDS], 0))
ppv = ens_all["se"] * pv / max(ens_all["se"] * pv + (1 - ens_all["sp"]) * (1 - pv), 1e-9)
npv = ens_all["sp"] * (1 - pv) / max(ens_all["sp"] * (1 - pv) + (1 - ens_all["se"]) * pv, 1e-9)
run.log(f"\n  전체 앙상블 동작점 — 외부 se {ens_all['se']:.3f} · sp {ens_all['sp']:.3f}")
run.log(f"    진짜 유병률 {pv:.3%} 로 재구성: PPV {ppv:.3%} · NPV {npv:.3%}")
run.log(f"    LR+ {ens_all['lrp']:.2f} · LR− {ens_all['lrn']:.3f}"
        "   ← 실험20b PTBDB 는 LR+ 1.23 / LR− 0.071 이었다")


In [ ]:
# CELL 7 — 사전등록 채점
run.log("\n" + "=" * 110)
run.log(f"【사전등록 채점】 시드 {len(SEEDS)}개 t-CI (t 배수가 3시드 4.303 → {len(SEEDS)}시드에서 내려간다)")
run.log("=" * 110)
V = {}

m1, l1, h1 = t_ci(DROP)
V["P-1"] = decide(l1, h1, DROP_THR, "<")
run.log(f"\n  P-1 외부 AUROC 낙폭 < {DROP_THR}")
run.log(f"      {m1:+.4f} [{l1:+.4f},{h1:+.4f}] → {MARK[V['P-1']]}")

lrns = [o["lrn"] for o in ENS]
m2, l2, h2 = t_ci_log(lrns)          # ★ 비율이므로 로그 스케일
V["P-2"] = decide(l2, h2, LRNEG_THR, "<")
run.log(f"\n  P-2 rule-out 이 살아남나 — 외부 LR− <= {LRNEG_THR}  (로그 스케일 CI)")
run.log(f"      LR− {m2:.3f} [{l2:.3f},{h2:.3f}] → {MARK[V['P-2']]}")
run.log(f"      참고 LR+ {np.exp(np.mean(np.log([o['lrp'] for o in ENS]))):.2f} "
        "— 1 에 가까우면 양성이어도 정보가 없다")

V["P-3"] = decide(ld, hd, D_THR, "<")
run.log(f"\n  P-3 외부 잔여결손 D <= {D_THR} (실험20 복제 — 12유도가 더 무너지나)")
run.log(f"      D {md:+.4f} [{ld:+.4f},{hd:+.4f}] → {MARK[V['P-3']]}")
run.log("      지지면 '유도를 줄이면 잃는다' 가 **두 번째 병원에서도** 성립하지 않는다")

V["P-4"] = decide(P4_LO, P4_HI, 0.0, ">")
run.log(f"\n  P-4 앙상블이 목표 민감도에 더 가깝게 착지하나 (차 > 0)")
run.log(f"      {P4_DIFF:+.4f} [{P4_LO:+.4f},{P4_HI:+.4f}] → {MARK[V['P-4']]}")
run.log("      ⚠️ 이 부트스트랩은 **레코드 표집**만 담는다 — '시드를 하나 더 뽑으면' 축은")
run.log("         못 본다. 시드 5개는 고정된 것으로 보고 읽어야 한다")

run.log("\n" + "=" * 110)
for k in ("P-1", "P-2", "P-3", "P-4"):
    run.log(f"  {k}: {MARK[V[k]] if V[k] is not None else MARK[None]}")
run.log("  ⚠️ SNOMED 라벨이라 **부위 국소화는 못 본다** — MI 유무 한 축이다")
run.log("  ⚠️ 내부는 전량모델의 내부 검증셋 · 외부는 같은 모델 → 낙폭은 과소추정")
run.log(f"  ⚠️ 대조군을 {N_CTRL}건으로 층화 표집했다. AUROC·se·sp 는 무관하지만 "
        "PPV 는 진짜 유병률로 재구성한 값이다")


In [ ]:
# CELL 8 — 그림
import matplotlib.pyplot as plt
fig, ax = plt.subplots(1, 3, figsize=(17, 4.4))

cs = list(LEADS); x = np.arange(len(cs)); w = 0.36
ax[0].bar(x - w/2, [np.mean(INT[c]) for c in cs], w, label="내부", color="#999999")
ax[0].bar(x + w/2, [np.mean(EXT[c]) for c in cs], w, label="외부", color="#d62728")
ax[0].set_xticks(x); ax[0].set_xticklabels(cs, fontsize=9)
ax[0].axhline(.5, ls=":", c="k", lw=1); ax[0].set_ylim(0, 1)
ax[0].set_ylabel("MI-any AUROC"); ax[0].legend(fontsize=8)
ax[0].set_title(f"외부 낙폭 · P-1 {MARK[V['P-1']]}")

ax[1].bar(["단일 시드(평균)", "앙상블"], [d_obs[0], d_obs[1]], color=["#ff7f0e", "#2ca02c"])
ax[1].set_ylabel(f"|달성 민감도 − {SENS_TARGET}|")
ax[1].set_title(f"동작점 이탈 · P-4 {MARK[V['P-4']]}")

ax[2].bar(["PTBDB(실험20b)", "Chapman/Ningbo"], [0.071, ens_all["lrn"]],
          color=["#bbbbbb", "#1f77b4"])
ax[2].axhline(LRNEG_THR, ls=":", c="r", lw=1)
ax[2].set_ylabel("LR−  (낮을수록 배제에 강함)")
ax[2].set_title(f"rule-out 이 살아남나 · P-2 {MARK[V['P-2']]}")
plt.tight_layout(); run.save_fig("exp21_chapman_ningbo", fig); plt.show()


In [ ]:
# CELL 9 — 결과 저장
res = {
    "week": 2, "exp_id": "exp21_chapman", "quest": "ailab-2026-0015",
    "step": "exp21-chapman-ningbo", "split": "inter",
    "task": "두 번째 외부 병원(Chapman-Shaoxing + Ningbo)에서 MI-any 복제 검정",
    "notebook": "notebooks/exp21_chapman_ningbo.ipynb",
    "metric": "external_auroc_mi_any", "value": round(float(np.mean(EXT[DEPLOY])), 4),
    "passed": bool(V["P-2"] is True),
    "n_records": len(RKEEP), "n_mi": int(YE.sum()),
    "prevalence_true": CONFIG["prevalence_true"], "mi_codes": CONFIG["mi_codes"],
    "deploy": DEPLOY, "seeds": SEEDS, "band_applied": CONFIG["band_applied"],
    "auroc_internal": {c: float(np.mean(INT[c])) for c in LEADS},
    "auroc_external": {c: float(np.mean(EXT[c])) for c in LEADS},
    "drop": [float(x) for x in DROP], "D_external": [float(x) for x in DE],
    "D_internal": [float(x) for x in DI],
    "operating_single": [{k: float(v) for k, v in o.items()} for o in SING],
    "operating_ensemble": [{k: float(v) for k, v in o.items()} for o in ENS],
    "p4_dev_single": float(d_obs[0]), "p4_dev_ensemble": float(d_obs[1]),
    "p4_diff_ci": [float(P4_LO), float(P4_HI)],
    "thr_single": [float(x) for x in TH_S], "thr_ensemble": float(TH_E),
    "ppv_at_true_prevalence": float(ppv), "npv_at_true_prevalence": float(npv),
    "lr_pos": float(ens_all["lrp"]), "lr_neg": float(ens_all["lrn"]),
    "verdicts": {k: V[k] for k in ("P-1", "P-2", "P-3", "P-4")},
    "caveats": [
        "SNOMED 라벨이라 부위 국소화 없음 — MI 유무 한 축",
        "대조군 층화 표집. PPV·NPV 는 인구조사의 진짜 유병률로 재구성한 값",
        "임계값은 내부에서 동결 — 외부 재조정 없음",
        "내부는 전량모델의 내부 검증셋이라 낙폭은 과소추정",
        "본 모델은 급성 심근경색이 아니라 판독 라벨을 예측한다"],
}
run.save_json("result", res); run.finish(res)
print(json.dumps({k: res[k] for k in ("metric", "value", "verdicts", "lr_neg",
                                      "p4_diff_ci", "n_mi")},
                 ensure_ascii=False, indent=2))
print("\n다음: python pipelines/ingest_run.py --results result.json "
      "--notebook notebooks/exp21_chapman_ningbo.ipynb")
